# Bayesian uncertainty: sampling the posterior instead of approximating it

[`01_inverse_problem.ipynb`](01_inverse_problem.ipynb) fits $\beta$ and $\alpha$ by least
squares and reads their uncertainty off the curvature of the objective at the optimum — the
Wald interval. Its own coverage study found that interval starting to fail exactly where
noise is realistic: at 10% measurement noise the nominal 95% interval covered the true
parameter only 88% of the time, not 95%.

This notebook answers the same question a different way: sample the actual posterior with an
ensemble MCMC sampler ([`emcee`](https://emcee.readthedocs.io/)), so a skewed or curved
credible region comes out skewed or curved instead of being forced into an ellipse. The price
is real - tens of thousands of forward integrations instead of a handful - and this notebook
does not hide that price.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from hiv_drc import (
    DRC_2020,
    estimate_parameters,
    generate_observations,
    log_likelihood,
    log_posterior,
    log_prior,
    plotting,
    run_mcmc,
    split_rhat,
)

print(f"true beta  = {DRC_2020.beta}")
print(f"true alpha = {DRC_2020.alpha}")

## 1. The same data as before

Same generator, same seed, same 5% proportional noise on $A$ and $T$ - so anything that
differs from the least-squares notebook is a property of the *method*, not the data.

In [ ]:
observations = generate_observations(DRC_2020, noise=0.05, seed=20260830)
names = ("beta", "alpha")

## 2. What the sampler actually evaluates

Three functions, each independently testable:

- `log_prior` - uniform on the rates, log-uniform on each series' relative noise level `eta`
- `log_likelihood` - Gaussian, with `sigma = eta * |model|` - the same proportional-noise model
  the generator itself uses, so on synthetic data this is the *correctly specified* likelihood
- `log_posterior = log_prior + log_likelihood`, skipping the (expensive) integration entirely
  when the prior already forbids the point

Unlike least squares, the noise level is not assumed via an ad hoc weighting - it is a fourth
and fifth unknown, `eta_A` and `eta_T`, inferred jointly with `beta` and `alpha`.

In [ ]:
theta_true = [DRC_2020.beta, DRC_2020.alpha, np.log(0.05), np.log(0.05)]
theta_bad = [0.4, 0.2, np.log(0.05), np.log(0.05)]

print(f"log_prior(truth)      = {log_prior(theta_true, names, observations):.3f}")
print(f"log_likelihood(truth) = {log_likelihood(theta_true, names, observations):.3f}")
print(f"log_posterior(truth)  = {log_posterior(theta_true, names, observations):.3f}")
print()
print(f"log_posterior(bad guess) = {log_posterior(theta_bad, names, observations):.3f}")
print("(much lower, as it should be)")

## 3. Sample

Walkers start jittered around the least-squares estimate rather than scattered across the
prior - `run_mcmc` computes that estimate internally as a starting point, not as the answer.

In [ ]:
posterior = run_mcmc(observations, fit=names, n_walkers=16, n_steps=800, burn=200, seed=20260830)
print(posterior.summary())

## 4. Did it converge?

Three independent checks, not one:

In [ ]:
fig = plotting.plot_trace(posterior)

A well-mixed chain looks like a fuzzy horizontal band with no single walker standing out from
the rest - that is what split-$\hat R$ near 1 is asserting numerically. `split_rhat` itself is
validated against two synthetic cases with a known answer before being trusted here:

In [ ]:
rng = np.random.default_rng(0)
mixed = rng.standard_normal((2000, 16))
offsets = rng.uniform(-5, 5, size=16)
stuck = offsets[None, :] + 0.1 * rng.standard_normal((2000, 16))

print(f"split_rhat of 16 independent N(0,1) chains : {split_rhat(mixed):.4f}  (should be ~1)")
print(f"split_rhat of 16 chains stuck at random offsets : {split_rhat(stuck):.1f}  (should be >> 1)")

## 5. The posterior itself

The corner plot below is the frequentist notebook's cost-surface contour, redrawn from actual
posterior draws instead of a grid of the objective. The same $(\beta, \alpha)$ correlation
shows up in both - reassuring, since they are two independent ways of asking the same question.

In [ ]:
fig = plotting.plot_posterior(posterior)

## 6. Does the model actually predict the data it was fit to?

A posterior *predictive* check draws parameter sets from the posterior, simulates each one,
and adds *that draw's own* fitted noise level - the resulting band is what the model expects a
new observation to look like, not just where the noise-free curve could sit. For a
well-specified model roughly 90% of the actual points should fall inside a 90% band.

In [ ]:
fig = plotting.plot_posterior_predictive(observations, posterior)

## 7. Frequentist versus Bayesian, on the same fit

Where the two intervals disagree, the local quadratic approximation behind the Wald interval
(or the fact that it does not know about the tighter, more defensible prior box
`bayesian.MCMC_BOUNDS` applies) was doing some of the work.

In [ ]:
fig = plotting.plot_bayes_vs_frequentist(posterior)

## 8. Does the credible interval calibrate better?

One run proves nothing about calibration - a 95% interval is a statement about long-run
frequency, not a guarantee for any single draw. The illustration below repeats the fit on a
handful of independent noise realisations at 10% noise, the level where the Wald interval's
own coverage study found the biggest gap. This is deliberately small (MCMC is not cheap); the
full 20-replicate measurement behind the README's table lives in
[`scripts/coverage_study.py`](../scripts/coverage_study.py) and takes several minutes to run.

In [ ]:
results = []
for seed in range(4):
    obs = generate_observations(noise=0.10, seed=3000 + seed)
    freq = estimate_parameters(obs)
    bayes = run_mcmc(obs, n_walkers=16, n_steps=500, burn=120, seed=seed)
    results.append((obs, freq, bayes))
    print(f"seed {seed}: "
          f"freq covers = {freq.covers_truth()}   "
          f"bayes covers = {bayes.covers_truth()}")

Four replicates is nowhere near enough to estimate a coverage *rate* - it is enough to see
that the two methods do not always agree, which is the point. See the README for the
full-scale comparison.

## What this notebook does not claim

The noise model here (`sigma = eta * |model|`, one `eta` per series) is exactly the model the
synthetic generator itself uses - on real data, where the true error process is unknown, that
match is not guaranteed, and a poorly specified noise model would bias the posterior just as a
wrong `baseline` value biases the least-squares fit (see the frequentist notebook's last
section). Bayesian inference makes assumptions explicit and inspectable; it does not remove
the need to get them right.